# Lekcja 13 – SQL: zapytania i łączenia

Rozwiązania zadań: **5, 6, 7, 8 i 18**.

Baza jest tworzona w pamięci, więc po zamknięciu połączenia nie zapisuje się na dysku.

In [1]:
import sqlite3
import time

import numpy as np
import pandas as pd

# Tworzymy bazę SQLite w pamięci.
conn = sqlite3.connect(":memory:")

## Zadanie 5 – LEFT JOIN

Używając tabel z zadania 4:

**Wymagania:**

- Wykonaj `LEFT JOIN` pokazujący wszystkich klientów (nawet bez zamówień).
- Użyj `COALESCE`, aby zamienić `NULL` na `0` w kwocie zamówienia.
- Znajdź klientów, którzy nie złożyli zamówienia (`WHERE order_id IS NULL`).

In [2]:
# Tabele customers i orders z zadania 4.
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "name": ["Jan Kowalski", "Anna Nowak", "Piotr Lis", "Maria Zając", "Tomasz Wójcik"],
    "city": ["Warszawa", "Kraków", "Gdańsk", "Wrocław", "Poznań"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106, 107],
    "customer_id": [1, 2, 1, 3, 1, 2, 3],
    "amount": [250.0, 120.5, 340.0, 80.0, 190.0, 450.0, 220.0]
})

# DataFrame'y pozostają bez zmian. Tworzymy z nich tabele w bazie SQL.
# if_exists="replace" pozwala ponownie uruchomić komórkę.
customers.to_sql("customers", conn, index=False, if_exists="replace")
orders.to_sql("orders", conn, index=False, if_exists="replace")

# LEFT JOIN zachowuje wszystkich klientów z lewej tabeli.
# Gdy klient nie ma zamówienia, COALESCE zamienia NULL na 0.
query_all_customers = """
SELECT
    c.customer_id,
    c.name,
    c.city,
    o.order_id,
    COALESCE(o.amount, 0) AS order_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_id
"""

all_customers = pd.read_sql_query(query_all_customers, conn)
print("Wszyscy klienci:")
print(all_customers)

# Brak dopasowanego order_id oznacza, że klient nie złożył zamówienia.
query_without_orders = """
SELECT
    c.customer_id,
    c.name,
    c.city
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
ORDER BY c.customer_id
"""

customers_without_orders = pd.read_sql_query(query_without_orders, conn)
print("\nKlienci bez zamówień:")
print(customers_without_orders)

Wszyscy klienci:
   customer_id           name      city  order_id  order_amount
0            1   Jan Kowalski  Warszawa     101.0         250.0
1            1   Jan Kowalski  Warszawa     103.0         340.0
2            1   Jan Kowalski  Warszawa     105.0         190.0
3            2     Anna Nowak    Kraków     102.0         120.5
4            2     Anna Nowak    Kraków     106.0         450.0
5            3      Piotr Lis    Gdańsk     104.0          80.0
6            3      Piotr Lis    Gdańsk     107.0         220.0
7            4    Maria Zając   Wrocław       NaN           0.0
8            5  Tomasz Wójcik    Poznań       NaN           0.0

Klienci bez zamówień:
   customer_id           name     city
0            4    Maria Zając  Wrocław
1            5  Tomasz Wójcik   Poznań


## Zadanie 6 – Podstawowy GROUP BY

Używając tabeli `orders`:

**Wymagania:**

- Policz liczbę zamówień dla każdego klienta.
- Oblicz łączną kwotę zamówień dla każdego klienta.
- Oblicz średnią wartość zamówienia dla każdego klienta.

In [3]:
# GROUP BY tworzy osobną grupę dla każdego customer_id.
query_order_summary = """
SELECT
    customer_id,
    COUNT(order_id) AS number_of_orders,
    SUM(amount) AS total_amount,
    ROUND(AVG(amount), 2) AS average_amount
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""

order_summary = pd.read_sql_query(query_order_summary, conn)
order_summary

,customer_id,number_of_orders,total_amount,average_amount
0,1,3,780.0,260.00
1,2,2,570.5,285.25
2,3,2,300.0,150.00


## Zadanie 7 – COUNT i SUM

**Wymagania:**

- Policz łączną liczbę produktów w tabeli `products`.
- Oblicz łączną wartość zapasów (`stock * price`) dla wszystkich produktów.
- Oblicz średnią cenę produktów w każdej kategorii.

In [4]:
# Tworzymy przykładową tabelę produktów.
products = pd.DataFrame({
    "product_id": [1, 2, 3, 4, 5],
    "name": ["Laptop", "Mysz", "Klawiatura", "Monitor", "Słuchawki"],
    "category": ["Elektronika", "Akcesoria", "Akcesoria", "Elektronika", "Akcesoria"],
    "price": [3500, 50, 200, 1200, 150],
    "stock": [10, 150, 80, 25, 60]
})

products.to_sql("products", conn, index=False, if_exists="replace")

# COUNT(*) liczy produkty, a SUM(stock * price) sumuje wartość ich zapasów.
query_products_total = """
SELECT
    COUNT(*) AS total_products,
    SUM(stock * price) AS total_inventory_value
FROM products
"""

products_total = pd.read_sql_query(query_products_total, conn)
print("Podsumowanie wszystkich produktów:")
print(products_total)

# AVG(price) oblicza średnią osobno dla każdej kategorii.
query_average_by_category = """
SELECT
    category,
    ROUND(AVG(price), 2) AS average_price
FROM products
GROUP BY category
ORDER BY category
"""

average_by_category = pd.read_sql_query(query_average_by_category, conn)
print("\nŚrednia cena w każdej kategorii:")
print(average_by_category)

Podsumowanie wszystkich produktów:
   total_products  total_inventory_value
0               5                  97500

Średnia cena w każdej kategorii:
      category  average_price
0    Akcesoria         133.33
1  Elektronika        2350.00


## Zadanie 8 – Tworzenie indeksu, EXPLAIN QUERY PLAN i pomiar czasu

**Wymagania:**

- Utwórz tabelę z 10 000 rekordami (użyj NumPy z L10).
- Zmierz czas zapytania `SELECT ... WHERE kolumna = ...` bez indeksu za pomocą `time.perf_counter()` (średnia z 5 powtórzeń).
- Uruchom `EXPLAIN QUERY PLAN` i zapisz plan bez indeksu (`SCAN`).
- Utwórz indeks na filtrowanej kolumnie (`CREATE INDEX`).
- Ponownie zmierz średni czas z 5 powtórzeń.
- Ponownie sprawdź plan (`SEARCH ... USING INDEX ...`).
- Porównaj czasy, przyspieszenie oraz plany.

In [5]:
# Osobna baza pozwala przeprowadzić czysty test indeksu.
conn_index = sqlite3.connect(":memory:")

np.random.seed(42)

# Każda wartość w kolumnie value występuje jeden raz.
large_table = pd.DataFrame({
    "record_id": np.arange(1, 10_001),
    "value": np.random.permutation(np.arange(1, 10_001)),
    "category": np.random.choice(["A", "B", "C"], 10_000)
})

large_table.to_sql("records", conn_index, index=False, if_exists="replace")

query = "SELECT * FROM records WHERE value = 7777"

# Plan i pomiar czasu BEZ indeksu.
plan_without_index = pd.read_sql_query(
    "EXPLAIN QUERY PLAN " + query,
    conn_index
)

times_without_index = []
for _ in range(5):
    start = time.perf_counter()
    result_without_index = pd.read_sql_query(query, conn_index)
    times_without_index.append(time.perf_counter() - start)

time_without_index = sum(times_without_index) / len(times_without_index)

# Tworzymy indeks na kolumnie używanej w WHERE.
conn_index.execute("CREATE INDEX idx_records_value ON records(value)")

# Plan i pomiar czasu Z indeksem.
plan_with_index = pd.read_sql_query(
    "EXPLAIN QUERY PLAN " + query,
    conn_index
)

times_with_index = []
for _ in range(5):
    start = time.perf_counter()
    result_with_index = pd.read_sql_query(query, conn_index)
    times_with_index.append(time.perf_counter() - start)

time_with_index = sum(times_with_index) / len(times_with_index)
speedup = time_without_index / time_with_index

print("Plan BEZ indeksu:")
print(plan_without_index)
print(f"\nŚredni czas BEZ indeksu: {time_without_index * 1000:.4f} ms")

print("\nPlan Z indeksem:")
print(plan_with_index)
print(f"\nŚredni czas Z indeksem: {time_with_index * 1000:.4f} ms")

print(f"\nPrzyspieszenie: {speedup:.2f}x")
print("\nRóżnica w planach:")
print("Bez indeksu:", plan_without_index["detail"].iloc[0])
print("Z indeksem:  ", plan_with_index["detail"].iloc[0])

Plan BEZ indeksu:
   id  parent  notused        detail
0   2       0      216  SCAN records

Średni czas BEZ indeksu: 1.0074 ms

Plan Z indeksem:
   id  parent  notused                                             detail
0   3       0       61  SEARCH records USING INDEX idx_records_value (...

Średni czas Z indeksem: 1.2766 ms

Przyspieszenie: 0.79x

Różnica w planach:
Bez indeksu: SCAN records
Z indeksem:   SEARCH records USING INDEX idx_records_value (value=?)


## Zadanie 18 – Self-JOIN

**Wymagania:**

- Utwórz tabelę `employees` z kolumnami: `employee_id`, `name`, `manager_id`.
- Użyj self-JOIN, aby wyświetlić pracownika i jego managera.
- Znajdź wszystkich pracowników bez managera (TOP management).
- Policz liczbę podwładnych dla każdego managera.

**Dataset:** własny (hierarchia).

In [6]:
# manager_id wskazuje employee_id przełożonego.
# None oznacza, że pracownik nie ma managera.
employees = pd.DataFrame({
    "employee_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": [
        "Anna Dyrektor",
        "Piotr Kierownik",
        "Marek Kierownik",
        "Ewa Programistka",
        "Jan Analityk",
        "Sara Programistka",
        "Adam Księgowy",
        "Lena Testerka"
    ],
    "manager_id": [None, 1, 1, 2, 2, 3, 3, 3]
})

employees.to_sql("employees", conn, index=False, if_exists="replace")

# Ta sama tabela występuje dwa razy: e1 to pracownik, e2 to manager.
query_employee_manager = """
SELECT
    e1.name AS employee,
    e2.name AS manager
FROM employees e1
LEFT JOIN employees e2 ON e1.manager_id = e2.employee_id
ORDER BY e1.employee_id
"""

employee_manager = pd.read_sql_query(query_employee_manager, conn)
print("Pracownicy i ich managerowie:")
print(employee_manager)

# Brak manager_id oznacza najwyższy poziom zarządzania.
query_top_management = """
SELECT
    employee_id,
    name
FROM employees
WHERE manager_id IS NULL
"""

top_management = pd.read_sql_query(query_top_management, conn)
print("\nPracownicy bez managera (TOP management):")
print(top_management)

# m oznacza managera, a e jego podwładnego.
query_subordinates = """
SELECT
    m.name AS manager,
    COUNT(e.employee_id) AS number_of_subordinates
FROM employees m
INNER JOIN employees e ON e.manager_id = m.employee_id
GROUP BY m.employee_id, m.name
ORDER BY number_of_subordinates DESC, m.name
"""

subordinates = pd.read_sql_query(query_subordinates, conn)
print("\nLiczba podwładnych każdego managera:")
print(subordinates)

Pracownicy i ich managerowie:
            employee          manager
0      Anna Dyrektor              NaN
1    Piotr Kierownik    Anna Dyrektor
2    Marek Kierownik    Anna Dyrektor
3   Ewa Programistka  Piotr Kierownik
4       Jan Analityk  Piotr Kierownik
5  Sara Programistka  Marek Kierownik
6      Adam Księgowy  Marek Kierownik
7      Lena Testerka  Marek Kierownik

Pracownicy bez managera (TOP management):
   employee_id           name
0            1  Anna Dyrektor

Liczba podwładnych każdego managera:
           manager  number_of_subordinates
0  Marek Kierownik                       3
1    Anna Dyrektor                       2
2  Piotr Kierownik                       2
